In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [4]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct", task="text-generation"
)

In [5]:
model = ChatHuggingFace(llm = llm)

In [6]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [7]:
def llm_qa(state:LLMState) -> LLMState:
    # extract the question from state
    question = state['question']

    # formm a prompt
    prompt = f"answer the following question {question}"

    # ask the question to llm
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [8]:
# define graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node("llm_qa", llm_qa)

# add edges
graph.add_edge(START,"llm_qa")
graph.add_edge("llm_qa",END)

# compile graph
workflow = graph.compile()

In [9]:
# execute graph
initial_state = {"question": "how far is moon from the earth?"}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'how far is moon from the earth?', 'answer': "The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. The closest point in the Moon's orbit, called perigee, can be as close as 363,300 kilometers (225,300 miles), while the farthest point, called apogee, can be as far as 405,500 kilometers (252,000 miles)."}
